# 11. pandas로 데이터 다루기

모은 데이터를 `for`와 딕셔너리로 고르고 세면 코드가 길어집니다. 표 모양 데이터를 다루는 라이브러리 **pandas**를 씁니다. DB 파트의 SQL로 하던 일(고르기 · 정렬 · 묶어서 계산)을 파이썬에서 합니다.

- 코드 칸 안의 `#>` 줄은 그 칸에서 할 일입니다
- 그 아래에 코드를 치고 Shift + Enter로 실행합니다

## 1. 불러오기

**기본 구조**

```
import pandas as pd

표 = pd.read_csv("주소 또는 파일.csv")   # CSV를 읽어 표로
표.head()                               # 앞 다섯 줄
표.shape                                # (줄 수, 칸 수)
```

pandas는 수업 시작에 `uv sync`로 설치했습니다(`ModuleNotFoundError: No module named 'pandas'`가 나면 터미널에서 `uv sync`).

`import 모듈 as 별명`은 모듈을 짧은 이름으로 부르는 모양입니다. pandas는 늘 `pd`로 씁니다. 표 하나를 **DataFrame**이라고 부릅니다.

1912년 타이타닉호 승객 891명의 기록입니다. 인터넷 주소의 CSV를 바로 읽습니다.

In [ ]:
#> pandas를 pd로 import하고, TITANIC_URL을 읽어 df에 넣고 앞 다섯 줄을 봅니다
TITANIC_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"

import pandas as pd

df = pd.read_csv(TITANIC_URL)


In [ ]:
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


| 칸 | 뜻 |
| --- | --- |
| `survived` | 살았으면 1, 아니면 0 |
| `pclass` | 객실 등급(1 · 2 · 3) |
| `sex` | 성별(`male` · `female`) |
| `age` | 나이 |
| `fare` | 요금 |
| `embark_town` | 탄 항구 |
| `who` | `man` · `woman` · `child` |
| `alone` | 혼자 탔으면 `True` |

In [ ]:
#> df의 줄 수와 칸 수(df.shape)를 확인합니다
df.shape

(891, 15)

## 2. 칸 살펴보기

**기본 구조**

```
표.info()                        # 칸마다 자료형 · 비어 있지 않은 값의 수
표["칸"]                          # 칸 하나
표[["칸1", "칸2"]]                # 칸 여러 개 (리스트로)
표["칸"].value_counts()          # 값마다 몇 개
표["칸"].mean()                  # 평균 · .max() · .min() · .sum()
```

In [ ]:
#> df.info()로 칸마다 자료형과 값의 수를 봅니다
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   survived     891 non-null    int64  
 1   pclass       891 non-null    int64  
 2   sex          891 non-null    str    
 3   age          714 non-null    float64
 4   sibsp        891 non-null    int64  
 5   parch        891 non-null    int64  
 6   fare         891 non-null    float64
 7   embarked     889 non-null    str    
 8   class        891 non-null    str    
 9   who          891 non-null    str    
 10  adult_male   891 non-null    bool   
 11  deck         203 non-null    str    
 12  embark_town  889 non-null    str    
 13  alive        891 non-null    str    
 14  alone        891 non-null    bool   
dtypes: bool(2), float64(2), int64(4), str(7)
memory usage: 92.4 KB


`age`는 891개 중 714개만 있습니다. 나이가 기록되지 않은 승객이 있습니다. 빈 값은 `NaN`으로 보이고, `.mean()` 같은 계산에서는 빼고 셉니다.

In [ ]:
#> "sex" · "age" · "survived" 세 칸만 골라 앞 다섯 줄을 봅니다
# SELECT sex, age, survived FROM df LIMIT 5;

df[['sex', 'age', 'survived']].head()

,sex,age,survived
0,male,22.0,0
1,female,38.0,1
2,female,26.0,1
3,female,35.0,1
4,male,35.0,0


In [ ]:
#> "survived" 칸의 값마다 몇 명인지 셉니다
df['survived'].value_counts()

survived
0    549
1    342
Name: count, dtype: int64

In [ ]:
#> "age" 칸의 평균과 가장 높은 나이(최댓값)를 print 한 번으로 출력합니다
print(df['age'].mean(), df['age'].max())

29.69911764705882 80.0


칸 하나에 대고 `.mean()`처럼 부르면 그 칸 전체를 계산합니다. `for`가 필요 없습니다.

## 3. 줄 고르기 · 정렬

**기본 구조**

```
표[표["칸"] == 값]                          # 조건에 맞는 줄만
표[(조건1) & (조건2)]                        # 둘 다 맞는 줄 (괄호 필수)
표.sort_values("칸", ascending=False)        # 큰 순서로
```

`df["sex"] == "female"`은 줄마다 `True` · `False`를 만듭니다. 그 결과를 `df[ ]`에 넣으면 `True`인 줄만 남습니다. SQL의 `WHERE`와 같은 일입니다.

In [ ]:
#> "sex"가 "female"인 줄만 골라 women에 넣고, 몇 명인지 확인합니다
# WHERE sex = 'female'
women = df[df['sex'] == 'female']
len(women)


314

In [ ]:
#> 1등급("pclass"가 1)이면서 여자인 줄만 골라 몇 명인지 확인합니다
df[(df['pclass'] == 1) & (df['sex'] == 'female')]

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
11,1,1,female,58.0,0,0,26.5500,S,First,woman,False,C,Southampton,yes,True
31,1,1,female,NaN,1,0,146.5208,C,First,woman,False,B,Cherbourg,yes,False
52,1,1,female,49.0,1,0,76.7292,C,First,woman,False,D,Cherbourg,yes,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
856,1,1,female,45.0,1,1,164.8667,S,First,woman,False,NaN,Southampton,yes,False
862,1,1,female,48.0,0,0,25.9292,S,First,woman,False,D,Southampton,yes,True
871,1,1,female,47.0,1,1,52.5542,S,First,woman,False,D,Southampton,yes,False
879,1,1,female,56.0,0,1,83.1583,C,First,woman,False,C,Cherbourg,yes,False


In [ ]:
#> "fare"가 큰 순서로 정렬해 "pclass" · "fare" · "survived" 칸의 앞 다섯 줄을 봅니다
# SELECT pclass, fare, survived FROM df
# ORDER BY fare DESC
# LIMIT 5
df.sort_values('fare', ascending=False)[['pclass', 'fare', 'survived']].head()

,pclass,fare,survived
679,1,512.3292,1
258,1,512.3292,1
737,1,512.3292,1
88,1,263.0000,1
438,1,263.0000,0
341,1,263.0000,1


## 4. 묶어서 계산

**기본 구조**

```
표["0과1인칸"].mean()                         # 1의 비율
표.groupby("칸")["계산할칸"].mean()            # 칸의 값마다 묶어서 평균
```

`survived`는 0과 1이라, 평균을 내면 **산 사람의 비율**(생존율)이 됩니다.

In [ ]:
#> 전체 생존율을 구합니다
df['survived'].mean()

np.float64(0.3838383838383838)

In [ ]:
#> 성별마다 생존율을 구합니다
df.groupby('sex')['survived'].mean()

sex
female    0.742038
male      0.188908
Name: survived, dtype: float64

`groupby`는 SQL의 `GROUP BY`와 같습니다. `sex`의 값마다 줄을 묶고, 묶음마다 `survived`의 평균을 냅니다.

In [ ]:
#> 객실 등급마다 생존율을 구합니다
df.groupby('pclass')['survived'].mean()

pclass
1    0.629630
2    0.472826
3    0.242363
Name: survived, dtype: float64

## 5. 두 칸으로 묶기 · 새 칸

**기본 구조**

```
표.groupby(["칸1", "칸2"])["계산할칸"].mean()   # 두 칸의 값마다 묶어서
표["새칸"] = 표["칸1"] + 표["칸2"]              # 줄마다 계산한 새 칸
```

In [ ]:
#> 성별과 객실 등급으로 함께 묶어 생존율을 구합니다
df.groupby(['sex', 'pclass'])['survived'].mean()

sex     pclass
female  1         0.968085
        2         0.921053
        3         0.500000
male    1         0.368852
        2         0.157407
        3         0.135447
Name: survived, dtype: float64

`sibsp`는 함께 탄 형제 · 배우자 수, `parch`는 함께 탄 부모 · 자녀 수입니다.

In [ ]:
#> "sibsp" + "parch"로 "family" 칸을 만들고, family마다 생존율을 구합니다
df['family'] = df['sibsp'] + df['parch']

In [ ]:
df.groupby('family')['survived'].mean()

family
0     0.303538
1     0.552795
2     0.578431
3     0.724138
4     0.200000
5     0.136364
6     0.333333
7     0.000000
10    0.000000
Name: survived, dtype: float64

## 6. CSV로 저장 · 읽기

**기본 구조**

```
표.to_csv("파일.csv", index=False)          # 저장 (index=False: 왼쪽 줄 번호는 빼고)
표 = pd.read_csv("파일.csv")                # 읽기
```

08장에서 `csv` 모듈로 읽으면 숫자도 모두 문자열이었습니다. pandas는 숫자 칸을 숫자로 읽습니다.

In [ ]:
#> 3절의 women을 women.csv로 저장하고, 다시 읽어 women2에 넣은 뒤 앞 세 줄을 봅니다
women.to_csv('women.csv')

In [ ]:
women2 = pd.read_csv('women.csv')

In [ ]:
#> women2의 "fare" 칸 평균을 구해 봅니다 (int() 없이)
women2['fare'].mean()

np.float64(44.47981783439491)

파일 목록에서 `women.csv`를 열어 봅니다. 구글 시트 · 엑셀에서도 표로 열립니다.

## 정리

| 할 일 | 방법 | SQL |
| --- | --- | --- |
| 읽기 | `pd.read_csv(주소)` · `.head()` · `.shape` · `.info()` | — |
| 칸 고르기 | `표["칸"]` · `표[["칸1", "칸2"]]` | `SELECT 칸1, 칸2` |
| 값마다 세기 | `표["칸"].value_counts()` | `GROUP BY 칸` + `COUNT(*)` |
| 칸 계산 | `.mean()` · `.max()` · `.min()` · `.sum()` | `AVG` · `MAX` · `MIN` · `SUM` |
| 줄 고르기 | `표[표["칸"] == 값]` · 여러 조건은 `(…) & (…)` | `WHERE` · `AND` |
| 정렬 | `표.sort_values("칸", ascending=False)` | `ORDER BY 칸 DESC` |
| 묶어서 계산 | `표.groupby("칸")["계산할칸"].mean()` · 두 칸은 `["칸1", "칸2"]` | `GROUP BY 칸` + `AVG(계산할칸)` |
| 새 칸 | `표["새칸"] = 표["칸1"] + 표["칸2"]` | — |
| CSV | `표.to_csv("파일.csv", index=False)` · `pd.read_csv("파일.csv")` | — |

| 에러 | 뜻 |
| --- | --- |
| `KeyError: '칸'` | 그런 칸 이름이 없음 — `df.columns`로 확인 |

연습: `exercises/11_pandas.ipynb`